In [1]:
import os
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import scanpy as sc
import anndata as ad
import hdf5plugin
import matplotlib.pyplot as plt
import seaborn as sns
import h5py

import squidpy as sq
import spatialdata as sd
from spatialdata_io import xenium
#import spatialdata_io as sio
import spatialdata_plot

/rfs/project/rfs-iCNyzSAaucw/nd590/miniconda3/envs/napari_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def print_sample_qc(adata):
    ## Print some stats
    
    fig, axs = plt.subplots(1, 4, figsize=(15, 4))

    axs[0].set_title("Total transcripts per cell")
    sns.histplot(
        adata.obs["total_counts"],
        kde=False,
        ax=axs[0],
    )
    
    axs[1].set_title("Unique transcripts per cell")
    sns.histplot(
        adata.obs["n_genes_by_counts"],
        kde=False,
        ax=axs[1],
    )
    
    
    axs[2].set_title("Area of segmented cells")
    sns.histplot(
        adata.obs["surface_area"],
        kde=False,
        ax=axs[2],
    )
    
    plt.show()

In [ ]:
xenium_path = '/home/nd590/rfs/rfs-teichlab-nfs-iCNyzSAaucw/nd590/projects/aging/proseg_results/'
xenium_files = pd.Series(os.listdir(xenium_path))
xenium_files_filtered = xenium_files[~xenium_files.str.contains('ipynb_checkpoints')]

In [ ]:
xenium_files_filtered

In [ ]:
adata_list = []

for file in xenium_files_filtered:
    ## Extract metadata from name
    sample_ID = file
    donor = file
    
    print(sample_ID)
    
    # 1. Load the count matrix generated by Proseg
    # This reads the counts.mtx.gz and the genes/barcodes if present
    sdata = sd.read_zarr(xenium_path + file + "/sdata.zarr")
    adata = sdata.tables["table"]

    adata.var_names = adata.var['gene']

    ## Update metadata
    adata.obs.index = sample_ID + '-' + adata.obs['cell'].astype(str)
    adata.obs['sample_barcode'] = adata.obs_names.tolist()
    adata.obs['sample_ID'] = sample_ID
    adata.obs['donor'] = donor

    ## Standard qc
    sc.pp.calculate_qc_metrics(adata, percent_top=(10, 20, 50, 150), inplace=True)

    ## Print some sample QC and show some plots
    print_sample_qc(adata)

    ## Some very mild filtering
    print('Total number of cells: ' + str(adata.obs.shape[0]))
    print('Fraction of cells with at least 30 genes detected:' + str(np.round((sum(adata.obs['n_genes_by_counts'] >= 30) /adata.obs.shape[0]), decimals=3))) 
    
    adata = adata[adata.obs['n_genes_by_counts'] >= 30].copy()

    adata_list.append(adata)

In [ ]:
xenium_adata = ad.concat(
    adata_list,
    axis=0,          # concatenate cells
    join="outer"    # union of genes 
)

In [ ]:
xenium_adata.obsm["X_spatial"] = xenium_adata.obsm["spatial"].copy()

In [ ]:
xenium_adata

In [ ]:
xenium_adata.obs["donor"].value_counts()

In [ ]:
sc.pl.violin(xenium_adata, groupby='sample_ID', keys=['n_genes_by_counts', 'pct_counts_in_top_50_genes'], size=0, rotation=90)

In [ ]:
xenium_adata.obs.index.name = None

In [ ]:
xenium_adata.write_h5ad('/home/nd590/rfs/rfs-teichlab-nfs-iCNyzSAaucw/nd590/projects/aging/objects/xenium_proseg_aging.h5ad')

### Integrate with sc reference 

### Load data

In [2]:
xenium_adata = ad.read_h5ad('/home/nd590/rfs/rfs-teichlab-nfs-iCNyzSAaucw/nd590/projects/aging/objects/xenium_proseg_aging.h5ad')

In [3]:
xenium_adata

AnnData object with n_obs × n_vars = 3531679 × 5051
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'region', 'sample_barcode', 'sample_ID', 'donor', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_150_genes'
    uns: 'sample_ID_colors'
    obsm: 'X_spatial', 'spatial'

In [4]:
xenium_adata.obs['chemistry_simple'] = 'xenium'
xenium_adata.obs['cells_spatial'] = 'spatial'
xenium_adata.layers["counts"] = xenium_adata.X.copy()

In [5]:
scell_adata = ad.read_h5ad('/home/nd590/rfs/rfs-teichlab-nfs-iCNyzSAaucw/vk348/thymus_ageing_atlas/General_analysis/data/objects/rna/thyAgeing_all_scvi_v6_2026-05-01.zarr')

In [6]:
scell_adata

AnnData object with n_obs × n_vars = 1028237 × 35530
    obs: 'barcode', 'sample', 'library', 'index', 'library_tcr_ab', 'library_tcr_gd', 'library_bcr', 'path_raw_gex', 'starsolo_version', 'path_cellbender_gex', 'path_cellranger_arc', 'donor', 'chemistry_simple', 'chemistry_detail', 'chemistry_orig', 'origin_institute', 'research_no', 'study', 'sort', 'sort_type', 'in_HTSA', 'public_data', 'type', 'cite', 'health_status', 'age_group', 'age_group2', 'age', 'age_num', 'age_unit', 'age_cont', 'sex', 'age_months', 'path_tcr_ab', 'path_tcr_gd', 'path_bcr', 'comment', 'n_counts', 'n_genes', 'percent_mito', 'percent_ribo', 'percent_hb', 'scrublet_score', 'predicted_doublet', 'anno_status', 'qc_status'
    var: 'gene_id', 'gene_name'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'chemistry_simple_colors', 'donor_colors', 'hvg', 'neighbors', 'sex_colors', 'taa_l1_colors', 'taa_l3_colors', 'umap'
    obsm: 'X_scVI', 'X_umap'
    obsp: 'connectivities', 'distances'

In [7]:
sc_anno = pd.read_csv('/home/nd590/rfs/rfs-teichlab-nfs-iCNyzSAaucw/vk348/thymus_ageing_atlas/General_analysis/data/objects/rna/thyAgeing_all_scvi_v5_2025-10-15_curatedAnno_v11.csv')

In [8]:
sc_anno

,Unnamed: 0,anno_status,qc_status,taa_l0,taa_l1,taa_l2,taa_l2_v2,taa_l3,taa_l3_v2,taa_l4,taa_l5
0,TA13072046-ACACCCTTCAGCACAT,include,PASS,Stroma,Fb,Fb,Fb,Fb-interlo,Fb-interlo,Fb-interlo,Fb-interlo
1,TA13072046-GCATGTACATGTAGTC,include,PASS,Stroma,Fb,Fb,Fb,Fb-interlo,Fb-interlo,Fb-interlo,Fb-interlo
2,TA13072046-CCGGTAGTCAGCTTAG,include,PASS,Stroma,Fb,Fb,Fb,Fb-interlo,Fb-interlo,Fb-interlo,Fb-interlo
3,TA13072046-GGCCGATCAGTCGTGC,include,PASS,Stroma,Fb,Fb,Fb,Fb-interlo,Fb-interlo,Fb-interlo,Fb-interlo
4,TA13072046-CGATCGGGTACGCTGC,include,PASS,Stroma,Fb,Fb,Fb,Fb-interlo,Fb-interlo,Fb-interlo,Fb-interlo
...,...,...,...,...,...,...,...,...,...,...,...
1025293,THY_81-GTTGAACAGGGTGAGG,include,PASS,Lymphoid,T,T_dev,T_DN,T_DN,T_DN,T_DN(Q),T_DN(Q)
1025294,THY_81-AACCTTTCAGTCGGAA,include,PASS,Lymphoid,T,T_dev,T_DN,T_DN,T_DN,T_DN(Q),T_DN(Q)
1025295,THY_81-ATGCATGAGAAGCGCT,include,PASS,Lymphoid,T,T_dev,T_DN,T_DN,T_DN,T_DN(Q),T_DN(Q)
1025296,THY_81-ATGGAGGGTAAGATTG,include,PASS,Lymphoid,T,T_dev,T_DN,T_DN,T_DN,T_DN(Q),T_DN(Q)


In [9]:
scell_adata_fil = scell_adata[
    (scell_adata.obs["age_group"] != "aged")
    & (scell_adata.obs_names.isin(sc_anno["Unnamed: 0"]))
].copy()

In [10]:
scell_adata_fil

AnnData object with n_obs × n_vars = 972869 × 35530
    obs: 'barcode', 'sample', 'library', 'index', 'library_tcr_ab', 'library_tcr_gd', 'library_bcr', 'path_raw_gex', 'starsolo_version', 'path_cellbender_gex', 'path_cellranger_arc', 'donor', 'chemistry_simple', 'chemistry_detail', 'chemistry_orig', 'origin_institute', 'research_no', 'study', 'sort', 'sort_type', 'in_HTSA', 'public_data', 'type', 'cite', 'health_status', 'age_group', 'age_group2', 'age', 'age_num', 'age_unit', 'age_cont', 'sex', 'age_months', 'path_tcr_ab', 'path_tcr_gd', 'path_bcr', 'comment', 'n_counts', 'n_genes', 'percent_mito', 'percent_ribo', 'percent_hb', 'scrublet_score', 'predicted_doublet', 'anno_status', 'qc_status'
    var: 'gene_id', 'gene_name'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'chemistry_simple_colors', 'donor_colors', 'hvg', 'neighbors', 'sex_colors', 'taa_l1_colors', 'taa_l3_colors', 'umap'
    obsm: 'X_scVI', 'X_umap'
    obsp: 'connectivities', 'distances'

In [11]:
scell_adata_fil.layers["counts"] = scell_adata_fil.X.copy()
scell_adata_fil.obs['cells_spatial'] = 'cells'
scell_adata_fil.obs.rename(columns={'sample': 'sample_ID'}, inplace=True)

### Harmonize genes

In [12]:
xenium_adata.obs_names = "xenium_" + xenium_adata.obs_names
scell_adata_fil.obs_names = "sc_" + scell_adata_fil.obs_names

In [13]:
common_genes = (
    xenium_adata.var_names
    .intersection(scell_adata_fil.var_names)
)

In [14]:
xenium_adata = xenium_adata[:, common_genes].copy()
scell_adata_fil = scell_adata_fil[:, common_genes].copy()

In [15]:
xenium_adata

AnnData object with n_obs × n_vars = 3531679 × 5043
    obs: 'cell', 'original_cell_id', 'centroid_x', 'centroid_y', 'centroid_z', 'component', 'volume', 'surface_area', 'scale', 'region', 'sample_barcode', 'sample_ID', 'donor', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_150_genes', 'chemistry_simple', 'cells_spatial'
    uns: 'sample_ID_colors'
    obsm: 'X_spatial', 'spatial'
    layers: 'counts'

In [16]:
scell_adata_fil

AnnData object with n_obs × n_vars = 972869 × 5043
    obs: 'barcode', 'sample_ID', 'library', 'index', 'library_tcr_ab', 'library_tcr_gd', 'library_bcr', 'path_raw_gex', 'starsolo_version', 'path_cellbender_gex', 'path_cellranger_arc', 'donor', 'chemistry_simple', 'chemistry_detail', 'chemistry_orig', 'origin_institute', 'research_no', 'study', 'sort', 'sort_type', 'in_HTSA', 'public_data', 'type', 'cite', 'health_status', 'age_group', 'age_group2', 'age', 'age_num', 'age_unit', 'age_cont', 'sex', 'age_months', 'path_tcr_ab', 'path_tcr_gd', 'path_bcr', 'comment', 'n_counts', 'n_genes', 'percent_mito', 'percent_ribo', 'percent_hb', 'scrublet_score', 'predicted_doublet', 'anno_status', 'qc_status', 'cells_spatial'
    var: 'gene_id', 'gene_name'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'chemistry_simple_colors', 'donor_colors', 'hvg', 'neighbors', 'sex_colors', 'taa_l1_colors', 'taa_l3_colors', 'umap'
    obsm: 'X_scVI', 'X_umap'
    layers: 'counts'
    obsp: 'connectivities', 'dis

In [17]:
scell_adata_fil.obs['chemistry_simple'].value_counts()

chemistry_simple
5GEX    765855
3GEX    207014
Name: count, dtype: int64

### Concatenate datasets

In [18]:
adata = ad.concat(
    [scell_adata_fil, xenium_adata],
    join="inner",
    label="cells_spatial",
    keys=["cells", "spatial"],
    fill_value=0
)

adata.obs["cells_spatial"].value_counts()

cells_spatial
spatial    3531679
cells       972869
Name: count, dtype: int64

In [19]:
adata

AnnData object with n_obs × n_vars = 4504548 × 5043
    obs: 'sample_ID', 'donor', 'chemistry_simple', 'cells_spatial'
    layers: 'counts'

In [20]:
adata.obs['donor'].value_counts()

donor
U48            1075573
A72             960559
U51             501863
Z17             430716
A67             223455
Z16             212145
A70             185277
A45             103028
TSC-CT10         37563
P2               36959
P3               35456
TSC-CT12         34940
NIHTHY0112       34699
TSC-CT8          33576
TSC-CT9          33536
TSC-CT13         32519
P1               31971
A53              30811
A66              30779
A43              28296
A16              27397
D5               20986
T06              20960
Z15              19117
D3               18947
baut_adult1      18934
A51              18769
NIHTHY0101       16959
Z13              16684
T07              16523
U40              16113
A74              15445
Z12              15286
A56              15264
D4               14210
D2               13789
U09              11367
D1                9704
Thy5              9309
Thy10             8900
baut_paed2        8607
Thy8              8330
Thy9              7742
T03  

In [21]:
adata.obs['chemistry_simple'].value_counts()

chemistry_simple
xenium    3531679
5GEX       765855
3GEX       207014
Name: count, dtype: int64

In [23]:
adata.write_h5ad("/home/nd590/rfs/rfs-teichlab-nfs-iCNyzSAaucw/nd590/projects/aging/objects/xenium_sc_sn_proseg_aging.h5ad")